In [ ]:
# this notebook conduct the following task:

# All the antigen pdb files are here \My Drive\@antibody\antigen_epitope\data\antigen_pdbs
# All the antigen obj files are here \My Drive\@antibody\antigen_epitope\data\antigen_objs
# From the pdb and obj files, generate the residue SI for each antigen in My Drive\@antibody\antigen_epitope\data\antigen_info_SI

In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive',force_remount=True)

In [ ]:
## for the first-time execution of this code
## install the packages in google Drive locally (/content/drive/MyDrive/Colab Notebooks/antigen_antibody/packages/)
import os
import sys

nb_path = '/content/notebooks'

if not os.path.exists(nb_path):
    os.symlink('/content/drive/MyDrive/@antibody/antigen_epitope/packages', nb_path)

sys.path.insert(0, nb_path)

In [ ]:
# first-time use execution
!pip install biopython --target=$nb_path

In [ ]:
!pip install scipy --target=$nb_path

In [ ]:
!pip install pywavefront --target=$nb_path

In [ ]:
!pip install ipyvolume --target=$nb_path

In [ ]:
!pip install pyvista --target=$nb_path

In [ ]:
!pip install matplotlib ipywidgets --target=$nb_path

In [ ]:
!apt-get install -y python3-pymol

In [ ]:
# INSTALL pymesh!!
# upload the .whl file for installation
# https://github.com/nuvolos-cloud/PyMesh/releases
from google.colab import files
uploaded = files.upload()
## install pymesh to calculate the curvature of the protein surface
!pip install pymesh2-0.3.1-cp310-cp310-linux_x86_64.whl --upgrade

In [ ]:
## non-first-time execution
## package was previously installed through process_data_positive_patch_3points-Graph.ipynb
import sys
sys.path.append('/content/drive/MyDrive/@antibody/antigen_epitope/packages')

In [ ]:
import os
import numpy as np
import pandas as pd
from Bio import PDB

In [ ]:
# dir to save info with SI
SI_output_dir = "/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_info_SI/"
antigen_obj_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_objs/'
antigen_pdb_dir = '/content/drive/MyDrive/@antibody/antigen_epitope/data/antigen_pdbs/'

In [ ]:
# Function to count files in a directory
def count_files(directory):
    return len([name for name in os.listdir(directory) if os.path.isfile(os.path.join(directory, name))])

# Count files in each directory
num_files_SI_output = count_files(SI_output_dir)
num_files_antigen_obj = count_files(antigen_obj_dir)
num_files_antigen_pdb = count_files(antigen_pdb_dir)

print(f"Number of files in SI output directory: {num_files_SI_output}")
print(f"Number of files in antigen object directory: {num_files_antigen_obj}")
print(f"Number of files in antigen PDB directory: {num_files_antigen_pdb}")

In [ ]:
import ipyvolume as ipv
from matplotlib import cm
import pyvista as pv
import pymesh
import pywavefront


In [ ]:
#@title Process .obj files
import pywavefront

for obj_filename in os.listdir(antigen_obj_dir):
  if obj_filename.endswith('.obj'):

    # Read the .obj file
    mesh = pywavefront.Wavefront(antigen_obj_dir + obj_filename, collect_faces=True)
    obj_vertices = set(mesh.vertices)
    # print('file name: ', obj_filename)
    # print('obj verices number: ', len(obj_vertices))
    obj_coords = np.array([v for v in set(obj_vertices)])
    obj_mean_curvature = np.array([np.nan] * len(obj_coords))
    obj_guassian_curvature = np.array([np.nan] * len(obj_coords))
    obj_curvature = np.array([np.nan] * len(obj_coords))


    # 15 seems to be the best neighbor number
    n_neighbors = 15

    # Calculate centroid by mean
    centroid = np.mean(obj_coords, axis=0)

    for idx in range(len(obj_coords)):
      current_coords = obj_coords[idx]
      dist_coords = obj_coords - current_coords

      dists = np.linalg.norm(dist_coords, axis=1)
      sorted_indices = np.argsort(dists)

      nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
      vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

      cloud = pv.PolyData(vertices)
      triangulated = cloud.delaunay_2d()
      shell = triangulated.extract_geometry()

      extracted_vertices = np.array(shell.points)
      extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
      extracted_faces = np.array([face for face in extracted_faces if 0 in face])

      mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
      mesh, info = pymesh.remove_isolated_vertices(mesh)

      mesh.add_attribute("vertex_mean_curvature")
      H = mesh.get_attribute("vertex_mean_curvature")
      mean_H = H[0]

      mesh.add_attribute("vertex_gaussian_curvature")
      K = mesh.get_attribute("vertex_gaussian_curvature")
      gaussian_K = K[0]

      elem = np.square(mean_H) - gaussian_K

      if elem < 0:
        elem = 1e-8
      elem = np.array(elem)

      k1 = mean_H + np.sqrt(elem)
      k2 = mean_H - np.sqrt(elem)

      si = (k1 + k2) / (k1 - k2)
      si = np.arctan(si) * (-2 / np.pi)

      # adjust curvature sign based on average direction to centroid
      dist_to_centroid = np.linalg.norm(current_coords - centroid)
      neighbor_dists_to_centroid = [np.linalg.norm(obj_coords[i] - centroid) for i in nearest_neighbors_idx]
      avg_neighbor_dist_to_centroid = np.mean(neighbor_dists_to_centroid)

      curvature_sign_by_distance = -1 if dist_to_centroid < avg_neighbor_dist_to_centroid else 1

      curvature_sign = curvature_sign_by_distance

      if curvature_sign > 0:
        si = abs(si)
      elif curvature_sign < 0:
        si = -abs(si)

      obj_mean_curvature[idx] = mean_H
      obj_guassian_curvature[idx] = gaussian_K
      obj_curvature[idx] = si


    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('protein', antigen_pdb_dir + obj_filename.replace('obj', 'pdb'))

    # select antigen chain name
    antigen_chain = obj_filename.split('_')[-2][0]
    model = structure[0]
    chain = model[antigen_chain]

    # Remove HETATM atom and select C-alpha
    atoms = [atom for atom in chain.get_atoms() if atom.get_parent().id[0] == " " and atom.name=='CA']
    atom_curvature = [None] * len(atoms)

    atom_coords = np.array([atom.coord for atom in atoms])
    # print('ATOM (CA) number: ', len(atoms))


    # Calculate the atom curvature based on the average curvature of nearest 10 points
    n_neighbors = 10
    for idx in range(len(atom_coords)):
      current_atom_coords = atom_coords[idx]
      dist_coords = obj_coords - current_atom_coords
      dists = np.linalg.norm(dist_coords, axis=1)

      nearest_neighbors_idx = np.argsort(dists)[:n_neighbors]
      atom_curvature[idx] = np.mean(obj_curvature[nearest_neighbors_idx])

    # dataframe
    columns = ['name', 'ResID', 'AA', 'SI']
    df = pd.DataFrame(columns=columns)
    df['ResID'] = [atom.get_full_id()[3][1] for atom in atoms]
    df['name'] = obj_filename.split('.')[0].replace('_antigen','')
    df['AA'] = [atom.get_parent().get_resname() for atom in atoms]
    df['SI'] = atom_curvature

    output_path = os.path.join(SI_output_dir, f"{obj_filename.split('.')[0]}_info_SI.tsv")
    df.to_csv(output_path, sep='\t', index=False)

    print(f"Processed {obj_filename} and saved to {output_path}")

In [ ]:
#@title Process .obj files again! for missing files

import pywavefront

# Ensure output directory exists
if not os.path.exists(SI_output_dir):
    os.makedirs(SI_output_dir)

for obj_filename in os.listdir(antigen_obj_dir):
    if obj_filename.endswith('.obj'):
        output_filename = f"{obj_filename.split('.')[0]}_info_SI.tsv"
        output_path = os.path.join(SI_output_dir, output_filename)

        # Skip processing if output file already exists
        if os.path.exists(output_path):
            print(f"Skipped {obj_filename} as it is already processed.")
            continue

        # Read the .obj file
        mesh = pywavefront.Wavefront(antigen_obj_dir + obj_filename, collect_faces=True)
        obj_vertices = set(mesh.vertices)
        obj_coords = np.array([v for v in obj_vertices])
        obj_mean_curvature = np.array([np.nan] * len(obj_coords))
        obj_guassian_curvature = np.array([np.nan] * len(obj_coords))
        obj_curvature = np.array([np.nan] * len(obj_coords))

        # Processing logic
        n_neighbors = 15
        centroid = np.mean(obj_coords, axis=0)

        for idx in range(len(obj_coords)):
            current_coords = obj_coords[idx]
            dist_coords = obj_coords - current_coords
            dists = np.linalg.norm(dist_coords, axis=1)
            nearest_neighbors_idx = np.argsort(dists)[1:(n_neighbors + 1)]
            vertices = dist_coords[np.concatenate([[idx], nearest_neighbors_idx]), :]

            cloud = pv.PolyData(vertices)
            triangulated = cloud.delaunay_2d()
            shell = triangulated.extract_geometry()

            extracted_vertices = np.array(shell.points)
            extracted_faces = np.array(shell.faces).reshape(-1, 4)[:, 1:]
            extracted_faces = np.array([face for face in extracted_faces if 0 in face])  # Convert to numpy array

            mesh = pymesh.form_mesh(extracted_vertices, extracted_faces)
            mesh, info = pymesh.remove_isolated_vertices(mesh)

            mesh.add_attribute("vertex_mean_curvature")
            H = mesh.get_attribute("vertex_mean_curvature")
            mean_H = H[0]

            mesh.add_attribute("vertex_gaussian_curvature")
            K = mesh.get_attribute("vertex_gaussian_curvature")
            gaussian_K = K[0]

            elem = np.square(mean_H) - gaussian_K
            if elem < 0:
                elem = 1e-8
            elem = np.array(elem)

            k1 = mean_H + np.sqrt(elem)
            k2 = mean_H - np.sqrt(elem)

            si = (k1 + k2) / (k1 - k2)
            si = np.arctan(si) * (-2 / np.pi)

            # Adjust curvature sign based on average direction to centroid
            dist_to_centroid = np.linalg.norm(current_coords - centroid)
            neighbor_dists_to_centroid = [np.linalg.norm(obj_coords[i] - centroid) for i in nearest_neighbors_idx]
            avg_neighbor_dist_to_centroid = np.mean(neighbor_dists_to_centroid)

            curvature_sign_by_distance = -1 if dist_to_centroid < avg_neighbor_dist_to_centroid else 1
            curvature_sign = curvature_sign_by_distance

            if curvature_sign > 0:
                si = abs(si)
            elif curvature_sign < 0:
                si = -abs(si)

            obj_mean_curvature[idx] = mean_H
            obj_guassian_curvature[idx] = gaussian_K
            obj_curvature[idx] = si

        # Parse PDB
        parser = PDB.PDBParser(QUIET=True)
        structure = parser.get_structure('protein', antigen_pdb_dir + obj_filename.replace('obj', 'pdb'))
        antigen_chain = obj_filename.split('_')[-2][0]
        model = structure[0]
        chain = model[antigen_chain]
        atoms = [atom for atom in chain.get_atoms() if atom.get_parent().id[0] == " " and atom.name == 'CA']
        atom_curvature = [None] * len(atoms)
        atom_coords = np.array([atom.coord for atom in atoms])

        # Calculate the atom curvature
        n_neighbors = 10
        for idx in range(len(atom_coords)):
            current_atom_coords = atom_coords[idx]
            dist_coords = obj_coords - current_atom_coords
            dists = np.linalg.norm(dist_coords, axis=1)

            nearest_neighbors_idx = np.argsort(dists)[:n_neighbors]
            atom_curvature[idx] = np.mean(obj_curvature[nearest_neighbors_idx])

        # Create DataFrame and save
        df = pd.DataFrame(columns=['name', 'ResID', 'AA', 'SI'])
        df['ResID'] = [atom.get_full_id()[3][1] for atom in atoms]
        df['name'] = obj_filename.split('.')[0].replace('_antigen', '')
        df['AA'] = [atom.get_parent().get_resname() for atom in atoms]
        df['SI'] = atom_curvature
        df.to_csv(output_path, sep='\t', index=False)
        print(f"Processed {obj_filename} and saved to {output_path}")

In [ ]:
# Count files in each directory
num_files_SI_output = count_files(SI_output_dir)
num_files_antigen_obj = count_files(antigen_obj_dir)
num_files_antigen_pdb = count_files(antigen_pdb_dir)

print(f"Number of files in SI output directory: {num_files_SI_output}")
print(f"Number of files in antigen object directory: {num_files_antigen_obj}")
print(f"Number of files in antigen PDB directory: {num_files_antigen_pdb}")